In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Getting started.pdf', 'sci.gslides', 'Swetha(32),Sarah(26),Jessica(12)_Informal Letter Practice 1.gdoc', 'Literature:).gdoc', 'Geography:).gdoc', 'C major_Swetha', 'IMG-20180614-WA0002.jpg', 'clarinet.jpg', 'IMG-20180413-WA0005.jpg', 'IMG_6122.PNG', 'farewell.mp4', '(no subject) - swetha.sudhak@gmail.com - Gmail.html', 'farewell edited.wlmp', 'farewell edited (1).mp4', 'farewell edited 2.mp4', 'rewrite the stars.html', 'Colorado Bar 100 to 121.MOV', 'IvP (1) (1).gslides', 'IvP (1).gslides', 'IvP (1).pptx', 'Allocation List Updated as of 26 August 2018.pdf', 'Untitled document (17).gdoc', '2018 PPCR Partnerships by Students.doc', 'f391451c-41d6-4202-8d2c-8ec532308963.JPG', 'Copy of f391451c-41d6-4202-8d2c-8ec532308963.JPG', 'PHOTO-2018-10-25-14-22-48 (1).jpg', 'PHOTO-2018-10-25-14-24-17 (1).jpg', 'PHOTO-2018-10-25-14-56-02.jpg', '4586c44d-d6f3-4b2c-bbb5-ed9fa9905df8.JPG', '99e907fd-f86e-4d28-975d-515865204013.JPG', '52eb6892-35fa-447f-ad85-d64a4076aee8.JPG', '8dbdb014-021b-4e88-a65c-f

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report
from datetime import datetime

# ── Config ───────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/TrafficImages"  # 👈 your path here
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-4
IMG_SIZE   = 224
DEVICE     = torch.device("cpu")
MODEL_PATH = "accident_classifier.pth"

# ── Transforms ────────────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Dataset ───────────────────────────────────────────────────────────────────
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transforms)
class_names  = full_dataset.classes
print(f"Classes found : {class_names}")
print(f"Total images  : {len(full_dataset)}")

total      = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size]
)

val_dataset.dataset  = datasets.ImageFolder(root=DATA_DIR, transform=val_transforms)
test_dataset.dataset = datasets.ImageFolder(root=DATA_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train : {train_size} images (70%)")
print(f"Val   : {val_size}   images (15%)")
print(f"Test  : {test_size}  images (15%)")
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

model     = build_model(num_classes=len(class_names))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ── Train / Eval functions ────────────────────────────────────────────────────
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), correct / len(loader.dataset), all_preds, all_labels

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\nTraining on {DEVICE}  |  {train_size} train  |  {val_size} val\n")
for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_epoch(model, train_loader)
    v_loss, v_acc, preds, labels_true = eval_epoch(model, val_loader)
    scheduler.step()
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={t_loss:.4f}  train_acc={t_acc:.4f}  "
          f"val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  "
          f"[{datetime.now().strftime('%H:%M:%S')}]")

# ── Report & Save ─────────────────────────────────────────────────────────────
print("\n── Validation Classification Report ──")
print(classification_report(labels_true, preds, target_names=class_names))

print("\n── Test Set Evaluation ──")
_, test_acc, test_preds, test_labels = eval_epoch(model, test_loader)
print(f"Test Accuracy : {test_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=class_names))

torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# SAVE PICKLE CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════
import pickle

PICKLE_PATH = "accident_classifier_checkpoint.pkl"
checkpoint  = {
    "model_state_dict": model.state_dict(),
    "class_names"     : class_names,
    "img_size"        : IMG_SIZE
}
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(checkpoint, f)
print(f"Checkpoint saved → {PICKLE_PATH}")


Classes found : ['accident', 'normal']
Total images  : 5399
Train : 3779 images (70%)
Val   : 809   images (15%)
Test  : 811  images (15%)

Training on cpu  |  3779 train  |  809 val

Epoch 01/10  train_loss=0.2180  train_acc=0.9100  val_loss=0.1268  val_acc=0.9468  [17:01:45]
Epoch 02/10  train_loss=0.1267  train_acc=0.9524  val_loss=0.0929  val_acc=0.9679  [17:19:58]
Epoch 03/10  train_loss=0.1091  train_acc=0.9590  val_loss=0.0970  val_acc=0.9617  [17:38:30]
Epoch 04/10  train_loss=0.0694  train_acc=0.9780  val_loss=0.0926  val_acc=0.9703  [17:56:52]
Epoch 05/10  train_loss=0.0545  train_acc=0.9794  val_loss=0.0966  val_acc=0.9666  [18:15:33]
Epoch 06/10  train_loss=0.0363  train_acc=0.9868  val_loss=0.0757  val_acc=0.9740  [18:33:33]
Epoch 07/10  train_loss=0.0309  train_acc=0.9892  val_loss=0.0672  val_acc=0.9753  [18:51:34]
Epoch 08/10  train_loss=0.0464  train_acc=0.9839  val_loss=0.0710  val_acc=0.9765  [19:09:32]
Epoch 09/10  train_loss=0.0296  train_acc=0.9913  val_loss=0.062

In [ ]:
def predict_image(image_path, model, class_names):
    from PIL import Image
    model.eval()
    img = val_transforms(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(img), dim=1)[0]
    for cls, p in zip(class_names, prob):
        print(f"  {cls}: {p:.2%}")
    return class_names[prob.argmax()]

# 👇 add your image path here
result = predict_image("/content/drive/MyDrive/TestingModel/Normal/01N.jpg", model, class_names)
print("Prediction:", result)

  accident: 3.44%
  normal: 96.56%
Prediction: normal


In [ ]:
import requests
from PIL import Image
from io import BytesIO

In [ ]:
def fetch_lta_cameras():
    url = "https://api.data.gov.sg/v1/transport/traffic-images"
    try:
        response = requests.get(url)
        data = response.json()
        cameras_raw = data["items"][0]["cameras"]
        cameras = {}
        for i, cam in enumerate(cameras_raw[:30], start=1):
            cam_id    = cam["camera_id"]
            lat       = cam["location"]["latitude"]
            lon       = cam["location"]["longitude"]
            image_url = cam["image"]
            cameras[i] = {
                "id"   : cam_id,
                "lat"  : lat,
                "lon"  : lon,
                "image": image_url,
                "label": f"Camera {cam_id} (Lat: {lat:.4f}, Lon: {lon:.4f})"
            }
        print("✅ LTA cameras loaded successfully.")
        return cameras
    except Exception as e:
        print(f"❌ Failed to fetch cameras: {e}")
        return {}

In [ ]:
test_image_bank = {
    # ACCIDENT IMAGES (A)
    1:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/01A.jpg",  "A"),
    2:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/02A.jpg",  "A"),
    3:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/03A.jpg",  "A"),
    4:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/04A.jpg",  "A"),
    5:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/05A.jpg",  "A"),
    6:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/06A.jpg",  "A"),
    7:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/07A.jpg",  "A"),
    8:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/08A.jpg",  "A"),
    9:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/09A.jpg",  "A"),
    10: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/10A.jpg", "A"),
    11: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/11A.jpg", "A"),
    12: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/12A.jpg", "A"),
    13: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/13A.jpg", "A"),
    14: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/14A.jpg", "A"),
    15: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/015A.jpg", "A"),
    # NO ACCIDENT IMAGES (NA)
    16: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/01N.jpg",  "NA"),
    17: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/02N.jpg",  "NA"),
    18: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/03N.jpg",  "NA"),
    19: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/04N.jpg",  "NA"),
    20: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/05N.jpg",  "NA"),
    21: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/06N.jpg",  "NA"),
    22: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/07N.jpg",  "NA"),
    23: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/08N.jpg",  "NA"),
    24: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/09N.jpg",  "NA"),
    25: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/10N.jpg", "NA"),
    26: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/11N.jpg", "NA"),
    27: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/12N.jpg", "NA"),
    28: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/13N.jpg", "NA"),
    29: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/14N.jpg", "NA"),
    30: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/15N.jpg", "NA"),
}

In [ ]:
def show_cameras(cameras):
    print("═" * 60)
    print("  LTA TRAFFIC CAMERA LOCATIONS")
    print("═" * 60)
    for num, cam in cameras.items():
        print(f"  {num:>2}. {cam['label']}")
    print("═" * 60)

In [ ]:
def show_image_bank():
    print("═" * 60)
    print("  TEST IMAGE BANK")
    print("  A = Accident  |  NA = No Accident")
    print("═" * 60)
    for num, (path, label) in test_image_bank.items():
        tag = "🚨 Accident   " if label == "A" else "✅ No Accident"
        print(f"  Image {num:>2} : {tag}")
    print("═" * 60)

In [ ]:
def run_prediction(image_path):
    if image_path.startswith("http"):
        response = requests.get(image_path)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(image_path).convert("RGB")

    img_tensor = val_transforms(img).unsqueeze(0).to(DEVICE)

    # accident detection
    model.eval()
    with torch.no_grad():
        acc_prob = torch.softmax(model(img_tensor), dim=1)[0]
    accident_pred = class_names[acc_prob.argmax()]
    accident_conf = acc_prob.max().item()

    return accident_pred, accident_conf, acc_prob

In [ ]:
def run_judge_system():
    # Step 1 — fetch and show cameras
    print("\nFetching LTA cameras...\n")
    cameras = fetch_lta_cameras()
    if not cameras:
        print("Could not load cameras. Check internet connection.")
        return

    show_cameras(cameras)
    cam_choice = int(input("\nEnter camera number (1-30): "))
    if cam_choice not in cameras:
        print("Invalid camera number.")
        return
    selected_camera = cameras[cam_choice]
    print(f"\n✅ Camera selected : {selected_camera['label']}")
    print(f"   Camera ID       : {selected_camera['id']}")
    print(f"   Location        : Lat {selected_camera['lat']}, Lon {selected_camera['lon']}")

    # Step 2 — show image bank and let judge choose
    print()
    show_image_bank()
    img_choice = int(input("\nEnter image number to test (1-30): "))
    if img_choice not in test_image_bank:
        print("Invalid image number.")
        return

    image_path, true_label = test_image_bank[img_choice]
    true_label_text = "Accident" if true_label == "A" else "No Accident"
    print(f"\n✅ Image {img_choice} selected  |  True label: {true_label_text}")

    # Step 3 — run model prediction
    print("\nRunning model prediction...\n")
    accident_pred, accident_conf, acc_prob = run_prediction(image_path)

    # Step 4 — output results
    print("═" * 60)
    print(f"  CAMERA LOCATION    : {selected_camera['label']}")
    print(f"  CAMERA ID          : {selected_camera['id']}")
    print(f"  IMAGE TESTED       : Image {img_choice}  [{true_label_text}]")
    print("─" * 60)
    print("  COLLISION DETECTION RESULTS")
    print("─" * 60)
    for cls, p in zip(class_names, acc_prob):
        bar = "█" * int(p.item() * 20)
        print(f"  {cls:<12} : {p:.2%}  {bar}")
    print("─" * 60)
    if accident_pred == "accident":
        print(f"  🚨 COLLISION DETECTED at {selected_camera['label']}")
        print(f"  CONFIDENCE         : {accident_conf:.2%}")
    else:
        print(f"  ✅ NO COLLISION at {selected_camera['label']}")
        print(f"  CONFIDENCE         : {accident_conf:.2%}")
    print("─" * 60)

    # Step 5 — correctness check
    predicted_label = "A" if accident_pred == "accident" else "NA"
    if predicted_label == true_label:
        print(f"  VERDICT : ✅ CORRECT  (Model predicted: {accident_pred.upper()})")
    else:
        print(f"  VERDICT : ❌ INCORRECT (True: {true_label_text} | Predicted: {accident_pred.upper()})")
    print("═" * 60)

# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════
run_judge_system()


Fetching LTA cameras...

✅ LTA cameras loaded successfully.
════════════════════════════════════════════════════════════
  LTA TRAFFIC CAMERA LOCATIONS
════════════════════════════════════════════════════════════
   1. Camera 9703 (Lat: 1.4229, Lon: 103.7730)
   2. Camera 1001 (Lat: 1.2953, Lon: 103.8711)
   3. Camera 1002 (Lat: 1.3195, Lon: 103.8786)
   4. Camera 1003 (Lat: 1.3240, Lon: 103.8729)
   5. Camera 1004 (Lat: 1.3195, Lon: 103.8751)
   6. Camera 1005 (Lat: 1.3635, Lon: 103.9054)
   7. Camera 1006 (Lat: 1.3571, Lon: 103.9020)
   8. Camera 1111 (Lat: 1.3654, Lon: 103.9540)
   9. Camera 1112 (Lat: 1.3605, Lon: 103.9614)
  10. Camera 1113 (Lat: 1.3170, Lon: 103.9886)
  11. Camera 1501 (Lat: 1.2741, Lon: 103.8513)
  12. Camera 1502 (Lat: 1.2714, Lon: 103.8618)
  13. Camera 1504 (Lat: 1.2941, Lon: 103.8761)
  14. Camera 1505 (Lat: 1.2753, Lon: 103.8664)
  15. Camera 1701 (Lat: 1.3236, Lon: 103.8588)
  16. Camera 1702 (Lat: 1.3436, Lon: 103.8602)
  17. Camera 1703 (Lat: 1.3281, Lo